# PicoClimate Test EDA

Detailed EDA with derivative and shapelet features.
Update the configuration cell if needed.

In [15]:
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA_DIR = Path("D:/repositories/personal/xai-spatio-temporal/data/picoclimate_test")
FIG_DIR = Path("D:/repositories/personal/xai-spatio-temporal/scripts/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

DATA_FILE = None  # optional: set to a specific CSV path
TIME_COL = None   # optional: set to known time column
GROUP_COLS = []   # optional: set to group keys, e.g., ["station_id"]
TARGET_COL = None # optional: exclude target from features/plots

PLOT_SAMPLE = 20000
MAX_NUM_COLS = 12

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams.update({
    "figure.figsize": (10, 6),
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "legend.fontsize": 11,
})

def timestamp():
    return datetime.now().strftime("%Y%m%d_%H%M%S")

def save_fig(fig, name):
    out_path = FIG_DIR / f"{name}_{timestamp()}.png"
    fig.tight_layout()
    fig.savefig(out_path, dpi=200)
    print(f"Saved: {out_path}")
    plt.close(fig)

def infer_time_col(columns):
    keys = ["time", "date", "timestamp"]
    for col in columns:
        lower = col.lower()
        if any(k in lower for k in keys):
            return col
    return None

def infer_group_cols(columns):
    keys = ["id", "station", "sensor", "device", "site"]
    return [col for col in columns if any(k in col.lower() for k in keys)]

In [16]:
csv_files = sorted(DATA_DIR.rglob("*.csv"))
print(f"Found {len(csv_files)} CSV files")
for f in csv_files:
    print(f)

if DATA_FILE is None:
    preferred_names = {"raw_measurements.csv", "window_features.csv"}
    preferred = [f for f in csv_files if f.name.lower() in preferred_names]
    DATA_FILE = preferred[0] if preferred else (csv_files[0] if csv_files else None)

print("Using:", DATA_FILE)

Found 2 CSV files
D:\repositories\personal\xai-spatio-temporal\data\picoclimate_test\raw_measurements.csv
D:\repositories\personal\xai-spatio-temporal\data\picoclimate_test\window_features.csv
Using: D:\repositories\personal\xai-spatio-temporal\data\picoclimate_test\raw_measurements.csv


In [17]:
if DATA_FILE is None:
    raise FileNotFoundError("No CSV file found in DATA_DIR.")

df = pd.read_csv(DATA_FILE)
print("Shape:", df.shape)
display(df.head())

Shape: (6000, 37)


,timestamp,date,day_index,time_slot,slot_index,location_id,city,lat,lon,elevation_m,...,no2_ppb,o3_ppb,noise_db,traffic_index,pedestrian_index,sky_view_factor,impervious_fraction,water_proximity,heat_index_c,missing_block_flag
0,2025-09-01 06:00:00+00:00,2025-09-01,0,morning,0,loc_000,Nantes,47.226932,-1.593119,59.391718,...,203.893741,153.967202,110.0,1.0,0.918,0.516913,0.2893,0.0,43.16,0
1,2025-09-01 12:00:00+00:00,2025-09-01,0,noon,1,loc_000,Nantes,47.226932,-1.593119,59.391718,...,205.681902,180.000000,110.0,1.0,0.527,0.516913,0.2893,0.0,45.86,0
2,2025-09-01 18:00:00+00:00,2025-09-01,0,evening,2,loc_000,Nantes,47.226932,-1.593119,59.391718,...,148.468470,83.461870,NaN,1.0,1.000,0.516913,0.2893,0.0,37.69,0
3,2025-09-01 23:00:00+00:00,2025-09-01,0,night,3,loc_000,Nantes,47.226932,-1.593119,59.391718,...,156.457866,NaN,64.4,1.0,1.000,0.516913,0.2893,0.0,38.62,0
4,2025-09-02 06:00:00+00:00,2025-09-02,1,morning,0,loc_000,Nantes,47.226932,-1.593119,59.391718,...,215.937880,146.486121,110.0,1.0,0.997,0.516913,NaN,0.0,42.41,0


In [18]:
display(df.sample(min(5, len(df)), random_state=42))
df.info()
missing = df.isna().mean().sort_values(ascending=False)
print("Top missing columns:")
display(missing.head(20))
print("Duplicate rows:", df.duplicated().sum())
display(df.describe(include="all").T)

,timestamp,date,day_index,time_slot,slot_index,location_id,city,lat,lon,elevation_m,...,no2_ppb,o3_ppb,noise_db,traffic_index,pedestrian_index,sky_view_factor,impervious_fraction,water_proximity,heat_index_c,missing_block_flag
1782,2025-09-26 18:00:00+00:00,2025-09-26,25,evening,2,loc_014,Nantes,47.184916,-1.535095,54.435385,...,62.699408,15.227641,103.8,1.000,NaN,0.746146,NaN,0.542458,37.59,0
3917,2025-09-20 12:00:00+00:00,2025-09-20,19,noon,1,loc_032,Nantes,47.207671,-1.498065,53.507475,...,220.000000,180.000000,110.0,0.247,1.0,0.706031,0.116289,0.591981,43.91,0
221,2025-09-26 12:00:00+00:00,2025-09-26,25,noon,1,loc_001,Montpellier,43.615480,3.865183,79.339603,...,NaN,NaN,NaN,NaN,NaN,NaN,0.729093,0.059902,45.36,1
2135,2025-09-24 23:00:00+00:00,2025-09-24,23,night,3,loc_017,Montpellier,43.601925,3.905226,39.773918,...,91.102188,68.634056,73.0,0.957,1.0,0.510359,0.461719,0.613909,36.85,0
5224,2025-09-17 06:00:00+00:00,2025-09-17,16,morning,0,loc_043,Montpellier,43.608153,3.824954,76.857359,...,NaN,180.000000,110.0,1.000,1.0,0.684609,0.117290,0.000000,40.74,0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6000 entries, 0 to 5999
Data columns (total 37 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   timestamp            6000 non-null   object 
 1   date                 6000 non-null   object 
 2   day_index            6000 non-null   int64  
 3   time_slot            6000 non-null   object 
 4   slot_index           6000 non-null   int64  
 5   location_id          6000 non-null   object 
 6   city                 6000 non-null   object 
 7   lat                  6000 non-null   float64
 8   lon                  6000 non-null   float64
 9   elevation_m          6000 non-null   float64
 10  hour                 6000 non-null   float64
 11  day_of_year          6000 non-null   float64
 12  true_regime          6000 non-null   object 
 13  air_temp_c           5590 non-null   float64
 14  rel_humidity_pct     5569 non-null   float64
 15  wind_speed_ms        5554 non-null   f

pm25_ugm3              0.133500
pedestrian_index       0.132833
pm10_ugm3              0.131333
no2_ppb                0.126333
o3_ppb                 0.126000
noise_db               0.125500
co2_ppm                0.124000
traffic_index          0.122500
solar_wm2              0.093667
ndvi                   0.090833
surface_temp_c         0.089833
heat_index_c           0.081167
longwave_wm2           0.077000
sky_view_factor        0.076500
impervious_fraction    0.076167
wind_speed_ms          0.074333
soil_moisture_pct      0.074000
wind_dir_deg           0.072667
rel_humidity_pct       0.071833
pressure_hpa           0.069167
dtype: float64

Duplicate rows: 0


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
timestamp,6000,120,2025-09-01 06:00:00+00:00,50,NaN,NaN,NaN,NaN,NaN,NaN,NaN
date,6000,30,2025-09-01,200,NaN,NaN,NaN,NaN,NaN,NaN,NaN
day_index,6000.0,NaN,NaN,NaN,14.5,8.656163,0.0,7.0,14.5,22.0,29.0
time_slot,6000,4,morning,1500,NaN,NaN,NaN,NaN,NaN,NaN,NaN
slot_index,6000.0,NaN,NaN,NaN,1.5,1.118127,0.0,0.75,1.5,2.25,3.0
location_id,6000,50,loc_000,120,NaN,NaN,NaN,NaN,NaN,NaN,NaN
city,6000,2,Nantes,3000,NaN,NaN,NaN,NaN,NaN,NaN,NaN
lat,6000.0,NaN,NaN,NaN,45.415109,1.796711,43.576118,43.61392,45.416431,47.210219,47.266804
lon,6000.0,NaN,NaN,NaN,1.157183,2.71309,-1.619238,-1.557182,1.163444,3.865183,3.987927
elevation_m,6000.0,NaN,NaN,NaN,65.099611,13.731979,39.773918,55.27177,64.477085,75.232444,95.139792


In [19]:
if TIME_COL is None:
    TIME_COL = infer_time_col(df.columns)
if not GROUP_COLS:
    GROUP_COLS = infer_group_cols(df.columns)

print("TIME_COL:", TIME_COL)
print("GROUP_COLS:", GROUP_COLS)

if TIME_COL:
    df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce")
    sort_cols = GROUP_COLS + [TIME_COL] if GROUP_COLS else [TIME_COL]
    df = df.sort_values(sort_cols)

TIME_COL: timestamp
GROUP_COLS: ['location_id', 'rel_humidity_pct']


In [20]:
numeric_cols = df.select_dtypes(include="number").columns.tolist()
if TARGET_COL in numeric_cols:
    numeric_cols.remove(TARGET_COL)
print("Numeric columns:", numeric_cols)

Numeric columns: ['day_index', 'slot_index', 'lat', 'lon', 'elevation_m', 'hour', 'day_of_year', 'air_temp_c', 'rel_humidity_pct', 'wind_speed_ms', 'wind_dir_deg', 'pressure_hpa', 'precipitation_mm', 'solar_wm2', 'longwave_wm2', 'surface_temp_c', 'soil_moisture_pct', 'ndvi', 'pm25_ugm3', 'pm10_ugm3', 'co2_ppm', 'no2_ppb', 'o3_ppb', 'noise_db', 'traffic_index', 'pedestrian_index', 'sky_view_factor', 'impervious_fraction', 'water_proximity', 'heat_index_c', 'missing_block_flag']


In [21]:
if numeric_cols:
    missing_pct = df[numeric_cols].isna().mean().sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(12, 6))
    missing_pct.head(30).plot(kind="bar", ax=ax)
    ax.set_title("Missingness (top 30 numeric columns)")
    ax.set_ylabel("Fraction missing")
    save_fig(fig, "missingness_numeric")

Saved: D:\repositories\personal\xai-spatio-temporal\scripts\figures\missingness_numeric_20260521_102018.png


In [22]:
if numeric_cols:
    cols = numeric_cols[:MAX_NUM_COLS]
    plot_df = df[cols]
    if len(plot_df) > PLOT_SAMPLE:
        plot_df = plot_df.sample(PLOT_SAMPLE, random_state=42)

    ncols = 3
    nrows = int(np.ceil(len(cols) / ncols))
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(14, 4 * nrows))
    axes = np.array(axes).reshape(-1)
    for ax, col in zip(axes, cols):
        sns.histplot(plot_df[col].dropna(), bins=40, ax=ax, kde=True)
        ax.set_title(col)
    for ax in axes[len(cols):]:
        ax.axis("off")
    save_fig(fig, "numeric_hist")

    fig, ax = plt.subplots(figsize=(12, 6))
    sns.boxplot(data=plot_df, orient="h", ax=ax)
    ax.set_title("Numeric column boxplots (sampled)")
    save_fig(fig, "numeric_boxplot")

Saved: D:\repositories\personal\xai-spatio-temporal\scripts\figures\numeric_hist_20260521_102021.png
Saved: D:\repositories\personal\xai-spatio-temporal\scripts\figures\numeric_boxplot_20260521_102025.png


In [23]:
if len(numeric_cols) >= 2:
    corr = df[numeric_cols].corr()
    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(corr, cmap="vlag", center=0, ax=ax)
    ax.set_title("Correlation heatmap")
    save_fig(fig, "correlation_heatmap")

Saved: D:\repositories\personal\xai-spatio-temporal\scripts\figures\correlation_heatmap_20260521_102026.png


In [24]:
if TIME_COL and numeric_cols:
    sample_cols = numeric_cols[: min(4, len(numeric_cols))]
    if GROUP_COLS:
        first_key = df[GROUP_COLS].drop_duplicates().iloc[0].to_list()
        mask = np.ones(len(df), dtype=bool)
        for col, val in zip(GROUP_COLS, first_key):
            mask &= df[col] == val
        ts_df = df.loc[mask, [TIME_COL] + sample_cols].sort_values(TIME_COL)
        title_suffix = " (first group)"
    else:
        ts_df = df[[TIME_COL] + sample_cols].sort_values(TIME_COL)
        title_suffix = ""

    if len(ts_df) > PLOT_SAMPLE:
        ts_df = ts_df.iloc[:PLOT_SAMPLE]

    fig, ax = plt.subplots(figsize=(12, 6))
    for col in sample_cols:
        ax.plot(ts_df[TIME_COL], ts_df[col], label=col)
    ax.set_title(f"Time series sample{title_suffix}")
    ax.set_xlabel(TIME_COL)
    ax.legend()
    save_fig(fig, "timeseries_sample")

Saved: D:\repositories\personal\xai-spatio-temporal\scripts\figures\timeseries_sample_20260521_102028.png


## Additional EDA
More distribution, missingness, outliers, pairplot, and rolling stats.

In [25]:
EXTRA_SAMPLE = 5000
PAIRPLOT_SAMPLE = 2000
MAX_CAT_COLS = 6
CAT_TOP_N = 12
MISSINGNESS_COLS = 30
ROLLING_WINDOW = 24

def safe_name(name):
    return "".join(ch if ch.isalnum() or ch in "-_" else "_" for ch in str(name))[:80]

row_count, col_count = df.shape
print(f"Rows: {row_count}, Columns: {col_count}")
print(f"Numeric cols: {len(numeric_cols)}")

cat_cols = [c for c in df.columns if c not in numeric_cols and c != TIME_COL]
if TARGET_COL in cat_cols:
    cat_cols.remove(TARGET_COL)
print("Categorical cols:", cat_cols)

if cat_cols:
    cardinality = df[cat_cols].nunique(dropna=True).sort_values(ascending=False)
    display(cardinality.head(20))

if GROUP_COLS:
    group_sizes = df.groupby(GROUP_COLS, dropna=False).size().sort_values(ascending=False)
    display(group_sizes.head(20))
    fig, ax = plt.subplots(figsize=(12, 4))
    group_sizes.head(20).plot(kind="bar", ax=ax)
    ax.set_title("Top group sizes")
    save_fig(fig, "group_sizes")

for col in cat_cols[:MAX_CAT_COLS]:
    vc = df[col].astype("string").value_counts(dropna=False).head(CAT_TOP_N)
    fig, ax = plt.subplots(figsize=(10, 4))
    vc.sort_values().plot(kind="barh", ax=ax)
    ax.set_title(f"Top categories: {col}")
    save_fig(fig, f"cat_{safe_name(col)}")

if TARGET_COL and TARGET_COL in df.columns:
    if pd.api.types.is_numeric_dtype(df[TARGET_COL]):
        fig, ax = plt.subplots(figsize=(10, 5))
        sns.histplot(df[TARGET_COL].dropna(), bins=40, ax=ax)
        ax.set_title(f"Target distribution: {TARGET_COL}")
        save_fig(fig, f"target_{safe_name(TARGET_COL)}")
    else:
        vc = df[TARGET_COL].astype("string").value_counts(dropna=False).head(CAT_TOP_N)
        fig, ax = plt.subplots(figsize=(10, 4))
        vc.sort_values().plot(kind="barh", ax=ax)
        ax.set_title(f"Target distribution: {TARGET_COL}")
        save_fig(fig, f"target_{safe_name(TARGET_COL)}")

if numeric_cols:
    miss_cols = df[numeric_cols].isna().mean().sort_values(ascending=False).head(MISSINGNESS_COLS).index
    miss_df = df[miss_cols]
    if len(miss_df) > EXTRA_SAMPLE:
        miss_df = miss_df.sample(EXTRA_SAMPLE, random_state=42)
    fig, ax = plt.subplots(figsize=(12, 6))
    sns.heatmap(miss_df.isna(), cbar=False, ax=ax)
    ax.set_title("Missingness heatmap (sampled)")
    save_fig(fig, "missingness_heatmap")

if numeric_cols:
    stats = pd.DataFrame({
        "mean": df[numeric_cols].mean(),
        "std": df[numeric_cols].std(),
        "skew": df[numeric_cols].skew(),
        "kurtosis": df[numeric_cols].kurtosis(),
    }).sort_values("skew", key=lambda s: s.abs(), ascending=False)
    display(stats.head(20))

    def outlier_rate(s):
        s = s.dropna()
        if len(s) < 4:
            return np.nan
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        if iqr == 0:
            return 0.0
        low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        return ((s < low) | (s > high)).mean()

    out_rates = df[numeric_cols].apply(outlier_rate).sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(12, 6))
    out_rates.head(30).plot(kind="bar", ax=ax)
    ax.set_title("Outlier rate (IQR) - top 30")
    ax.set_ylabel("Fraction outliers")
    save_fig(fig, "outlier_rate")

if numeric_cols:
    pair_cols = numeric_cols[: min(5, len(numeric_cols))]
    pair_df = df[pair_cols].dropna()
    if len(pair_df) > PAIRPLOT_SAMPLE:
        pair_df = pair_df.sample(PAIRPLOT_SAMPLE, random_state=42)
    if len(pair_df) > 1:
        g = sns.pairplot(pair_df, corner=True, diag_kind="hist", plot_kws={"s": 12, "alpha": 0.4})
        save_fig(g.fig, "pairplot_sample")

if TIME_COL and numeric_cols:
    from pandas.plotting import autocorrelation_plot
    col = numeric_cols[0]
    ts_df = df[[TIME_COL, col]].dropna().sort_values(TIME_COL)
    if len(ts_df) > PLOT_SAMPLE:
        ts_df = ts_df.iloc[:PLOT_SAMPLE]
    if len(ts_df) > 1:
        fig, ax = plt.subplots(figsize=(10, 4))
        autocorrelation_plot(ts_df[col], ax=ax)
        ax.set_title(f"Autocorrelation: {col}")
        save_fig(fig, f"autocorr_{safe_name(col)}")

        roll = ts_df[col].rolling(ROLLING_WINDOW, min_periods=max(2, ROLLING_WINDOW // 4))
        fig, ax = plt.subplots(figsize=(12, 6))
        ax.plot(ts_df[TIME_COL], ts_df[col], alpha=0.35, label="value")
        ax.plot(ts_df[TIME_COL], roll.mean(), label=f"rolling mean ({ROLLING_WINDOW})")
        ax.plot(ts_df[TIME_COL], roll.std(), label=f"rolling std ({ROLLING_WINDOW})")
        ax.set_title(f"Rolling stats: {col}")
        ax.legend()
        save_fig(fig, f"rolling_stats_{safe_name(col)}")

Rows: 6000, Columns: 37
Numeric cols: 31
Categorical cols: ['date', 'time_slot', 'location_id', 'city', 'true_regime']


location_id    50
date           30
time_slot       4
true_regime     4
city            2
dtype: int64

location_id  rel_humidity_pct
loc_002      5.0                 117
loc_044      5.0                 116
loc_020      5.0                 115
loc_030      5.0                 115
loc_036      5.0                 114
loc_001      5.0                 114
loc_011      5.0                 114
loc_022      5.0                 114
loc_009      5.0                 114
loc_006      5.0                 114
loc_033      5.0                 114
loc_014      5.0                 113
loc_047      5.0                 112
loc_017      5.0                 112
loc_026      5.0                 112
loc_028      5.0                 112
loc_048      5.0                 111
loc_039      5.0                 111
loc_029      5.0                 111
loc_003      5.0                 111
dtype: int64

Saved: D:\repositories\personal\xai-spatio-temporal\scripts\figures\group_sizes_20260521_102029.png
Saved: D:\repositories\personal\xai-spatio-temporal\scripts\figures\cat_date_20260521_102030.png
Saved: D:\repositories\personal\xai-spatio-temporal\scripts\figures\cat_time_slot_20260521_102030.png
Saved: D:\repositories\personal\xai-spatio-temporal\scripts\figures\cat_location_id_20260521_102031.png
Saved: D:\repositories\personal\xai-spatio-temporal\scripts\figures\cat_city_20260521_102031.png
Saved: D:\repositories\personal\xai-spatio-temporal\scripts\figures\cat_true_regime_20260521_102032.png
Saved: D:\repositories\personal\xai-spatio-temporal\scripts\figures\missingness_heatmap_20260521_102033.png


,mean,std,skew,kurtosis
air_temp_c,49.951424,0.564495,-15.785209,288.314836
rel_humidity_pct,5.301059,1.905415,8.467081,84.357274
missing_block_flag,0.059833,0.237198,3.712630,11.787554
pedestrian_index,0.816405,0.297301,-1.541523,1.062535
traffic_index,0.835688,0.265358,-1.441743,0.712773
no2_ppb,174.272917,57.961307,-1.203173,0.376343
co2_ppm,878.397070,583.180927,1.037367,0.169512
pm10_ugm3,57.616495,56.956490,0.988422,0.528846
wind_speed_ms,19.956900,5.006890,-0.981077,0.322864
wind_dir_deg,72.782393,71.473320,0.973901,0.342020


Saved: D:\repositories\personal\xai-spatio-temporal\scripts\figures\outlier_rate_20260521_102034.png
Saved: D:\repositories\personal\xai-spatio-temporal\scripts\figures\pairplot_sample_20260521_102037.png
Saved: D:\repositories\personal\xai-spatio-temporal\scripts\figures\autocorr_day_index_20260521_102039.png
Saved: D:\repositories\personal\xai-spatio-temporal\scripts\figures\rolling_stats_day_index_20260521_102040.png


## Distribution report
Target distribution, feature distributions, log checks, and bivariate views.

In [ ]:
DIST_MAX_NUM_COLS = 8
DIST_MAX_CAT_COLS = 6
TARGET_TOP_N = 20
BIVAR_MAX_NUM = 4
BIVAR_MAX_CAT = 4
BIVAR_SAMPLE = 5000
SKEW_THRESHOLD = 1.0

def is_classification_target(series):
    if pd.api.types.is_object_dtype(series) or pd.api.types.is_bool_dtype(series) or pd.api.types.is_categorical_dtype(series):
        return True
    return series.nunique(dropna=True) <= TARGET_TOP_N

def plot_target_distribution(df_in, target_col):
    if not target_col or target_col not in df_in.columns:
        print("TARGET_COL not set; skipping target plots.")
        return
    s = df_in[target_col]
    if is_classification_target(s):
        vc = s.astype("string").value_counts(dropna=False)
        total = vc.sum()
        top = vc.head(TARGET_TOP_N)
        fig, ax = plt.subplots(figsize=(10, 5))
        top.sort_values().plot(kind="barh", ax=ax)
        for p in ax.patches:
            pct = 100.0 * p.get_width() / total if total else 0.0
            ax.text(p.get_width() + 0.01 * total, p.get_y() + p.get_height() / 2, f"{pct:.1f}%")
        ax.set_title(f"Target distribution (classification): {target_col}")
        save_fig(fig, f"target_class_{safe_name(target_col)}")
    else:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        sns.histplot(s.dropna(), bins=40, kde=True, ax=axes[0])
        sns.boxplot(x=s.dropna(), ax=axes[1])
        axes[0].set_title(f"Target histogram: {target_col}")
        axes[1].set_title("Target boxplot")
        save_fig(fig, f"target_reg_{safe_name(target_col)}")

def numeric_summary_table(df_in, cols):
    if not cols:
        return
    summary = df_in[cols].describe().T
    summary["skew"] = df_in[cols].skew()
    summary["kurtosis"] = df_in[cols].kurtosis()
    display(summary.head(20))

def plot_numeric_distributions(df_in, cols, max_cols):
    for col in cols[:max_cols]:
        s = df_in[col].dropna()
        if s.empty:
            continue
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        sns.histplot(s, bins=40, kde=True, ax=axes[0])
        sns.boxplot(x=s, ax=axes[1])
        axes[0].set_title(f"{col} (skew={s.skew():.2f})")
        axes[1].set_title("Boxplot")
        save_fig(fig, f"dist_{safe_name(col)}")

def log1p_shift(series):
    s = series.dropna()
    if s.empty:
        return s, 0.0
    min_val = s.min()
    shift = -min_val + 1 if min_val <= -1 else 0.0
    return np.log1p(s + shift), shift

def plot_log_transform_comparison(df_in, cols, threshold):
    for col in cols:
        s = df_in[col].dropna()
        if s.empty:
            continue
        skew = s.skew()
        if abs(skew) < threshold:
            continue
        logged, shift = log1p_shift(s)
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        sns.histplot(s, bins=40, kde=True, ax=axes[0])
        sns.histplot(logged, bins=40, kde=True, ax=axes[1])
        axes[0].set_title(f"{col} original")
        axes[1].set_title(f"log1p (shift={shift:.2f})")
        save_fig(fig, f"log_compare_{safe_name(col)}")

def plot_bivariate_with_target(df_in, target_col):
    if not target_col or target_col not in df_in.columns:
        return
    if is_classification_target(df_in[target_col]):
        for col in numeric_cols[:BIVAR_MAX_NUM]:
            if col == target_col:
                continue
            sample = df_in[[col, target_col]].dropna()
            if len(sample) > BIVAR_SAMPLE:
                sample = sample.sample(BIVAR_SAMPLE, random_state=42)
            if sample.empty:
                continue
            fig, ax = plt.subplots(figsize=(10, 5))
            sns.violinplot(data=sample, x=target_col, y=col, inner="quartile", cut=0, ax=ax)
            ax.set_title(f"{col} by {target_col}")
            ax.tick_params(axis="x", rotation=30)
            save_fig(fig, f"bivar_{safe_name(col)}_by_{safe_name(target_col)}")

        for col in cat_cols[:BIVAR_MAX_CAT]:
            ct = pd.crosstab(df_in[col].astype("string"), df_in[target_col].astype("string"), normalize="index")
            if ct.empty:
                continue
            top = df_in[col].astype("string").value_counts().head(CAT_TOP_N).index
            ct = ct.loc[ct.index.intersection(top)]
            fig, ax = plt.subplots(figsize=(10, 5))
            ct.plot(kind="bar", stacked=True, ax=ax)
            ax.set_title(f"{col} vs {target_col} (row %)")
            ax.legend(title=target_col, bbox_to_anchor=(1.02, 1), loc="upper left")
            save_fig(fig, f"bivar_{safe_name(col)}_stacked")
    else:
        for col in numeric_cols[:BIVAR_MAX_NUM]:
            if col == target_col:
                continue
            sample = df_in[[col, target_col]].dropna()
            if len(sample) > BIVAR_SAMPLE:
                sample = sample.sample(BIVAR_SAMPLE, random_state=42)
            if sample.empty:
                continue
            fig, ax = plt.subplots(figsize=(10, 5))
            sns.scatterplot(data=sample, x=col, y=target_col, s=10, alpha=0.4, ax=ax)
            ax.set_title(f"{target_col} vs {col}")
            save_fig(fig, f"bivar_{safe_name(target_col)}_vs_{safe_name(col)}")

        for col in cat_cols[:BIVAR_MAX_CAT]:
            sample = df_in[[col, target_col]].dropna()
            if sample.empty:
                continue
            top = sample[col].astype("string").value_counts().head(CAT_TOP_N).index
            sample = sample[sample[col].astype("string").isin(top)]
            fig, ax = plt.subplots(figsize=(12, 5))
            sns.boxplot(data=sample, x=col, y=target_col, ax=ax)
            ax.tick_params(axis="x", rotation=30)
            ax.set_title(f"{target_col} by {col}")
            save_fig(fig, f"bivar_{safe_name(target_col)}_by_{safe_name(col)}")

        if target_col in numeric_cols:
            corr = df_in[numeric_cols].corrwith(df_in[target_col]).dropna().sort_values()
            fig, ax = plt.subplots(figsize=(10, 6))
            corr.tail(20).plot(kind="barh", ax=ax)
            ax.set_title(f"Correlation with {target_col} (top 20)")
            save_fig(fig, f"corr_with_{safe_name(target_col)}")

if numeric_cols:
    row_missing = df[numeric_cols].isna().mean(axis=1)
    fig, ax = plt.subplots(figsize=(10, 4))
    sns.histplot(row_missing, bins=40, ax=ax)
    ax.set_title("Missingness per row (numeric)")
    save_fig(fig, "missingness_row")

plot_target_distribution(df, TARGET_COL)
numeric_summary_table(df, numeric_cols)
plot_numeric_distributions(df, numeric_cols, DIST_MAX_NUM_COLS)
plot_log_transform_comparison(df, numeric_cols[:DIST_MAX_NUM_COLS], SKEW_THRESHOLD)
plot_bivariate_with_target(df, TARGET_COL)

In [26]:
DERIVATIVE_COLS = numeric_cols  # override if you want fewer columns

def add_derivatives(df_in, cols, group_cols, time_col):
    df_out = df_in.copy()
    if time_col:
        sort_cols = group_cols + [time_col] if group_cols else [time_col]
        df_out = df_out.sort_values(sort_cols)

    if group_cols:
        g = df_out.groupby(group_cols, sort=False, dropna=False)
        for col in cols:
            df_out[f"{col}__d1"] = g[col].diff()
            df_out[f"{col}__d2"] = df_out.groupby(group_cols, sort=False, dropna=False)[f"{col}__d1"].diff()
            df_out[f"{col}__d3"] = df_out.groupby(group_cols, sort=False, dropna=False)[f"{col}__d2"].diff()
    else:
        for col in cols:
            df_out[f"{col}__d1"] = df_out[col].diff()
            df_out[f"{col}__d2"] = df_out[f"{col}__d1"].diff()
            df_out[f"{col}__d3"] = df_out[f"{col}__d2"].diff()
    return df_out

df_deriv = add_derivatives(df, DERIVATIVE_COLS, GROUP_COLS, TIME_COL)
print("Derivative features added:", [c for c in df_deriv.columns if c.endswith("__d1")][:5])

Derivative features added: ['day_index__d1', 'slot_index__d1', 'lat__d1', 'lon__d1', 'elevation_m__d1']


In [27]:
SHAPELET_SOURCE_COL = None  # set to a numeric column for shapelets
SHAPELET_LENGTH = 30
N_SHAPELETS = 6
MAX_GROUPS = 50
MAX_POINTS_PER_SERIES = 2000
RANDOM_SEED = 42

def z_normalize(x, eps=1e-8):
    x = np.asarray(x, dtype=float)
    m = np.nanmean(x)
    s = np.nanstd(x)
    if s < eps:
        return x * 0.0
    return (x - m) / s

def min_distance_to_shapelet(series, shapelet):
    if len(series) < len(shapelet):
        return np.nan
    series = z_normalize(series)
    try:
        windows = np.lib.stride_tricks.sliding_window_view(series, len(shapelet))
    except AttributeError:
        windows = np.array([series[i : i + len(shapelet)] for i in range(len(series) - len(shapelet) + 1)])
    dists = np.linalg.norm(windows - shapelet, axis=1)
    return float(np.min(dists))

shapelet_df = None
if numeric_cols:
    if SHAPELET_SOURCE_COL is None:
        SHAPELET_SOURCE_COL = numeric_cols[0]

    rng = np.random.default_rng(RANDOM_SEED)
    series_list = []
    group_keys = []

    if GROUP_COLS:
        groups = df_deriv.groupby(GROUP_COLS, sort=False, dropna=False)
        keys = list(groups.groups.keys())[:MAX_GROUPS]
        for key in keys:
            if isinstance(key, tuple):
                if any(pd.isna(v) for v in key):
                    continue
                mask = np.ones(len(df_deriv), dtype=bool)
                for col, val in zip(GROUP_COLS, key):
                    mask &= df_deriv[col].eq(val)
                gdf = df_deriv.loc[mask]
            else:
                if pd.isna(key):
                    continue
                gdf = df_deriv[df_deriv[GROUP_COLS[0]].eq(key)]
            series = gdf[SHAPELET_SOURCE_COL].to_numpy()
            series = pd.Series(series).interpolate(limit_direction="both").to_numpy()
            series = series[:MAX_POINTS_PER_SERIES]
            if len(series) >= SHAPELET_LENGTH:
                series_list.append(series)
                group_keys.append(key)
    else:
        series = df_deriv[SHAPELET_SOURCE_COL].to_numpy()
        series = pd.Series(series).interpolate(limit_direction="both").to_numpy()
        series = series[:MAX_POINTS_PER_SERIES]
        if len(series) >= SHAPELET_LENGTH:
            series_list.append(series)
            group_keys.append(None)

    if series_list:
        shapelets = []
        for _ in range(N_SHAPELETS):
            s = series_list[rng.integers(len(series_list))]
            start = rng.integers(0, len(s) - SHAPELET_LENGTH + 1)
            shapelets.append(z_normalize(s[start : start + SHAPELET_LENGTH]))

        feature_rows = []
        for series, key in zip(series_list, group_keys):
            feats = [min_distance_to_shapelet(series, shp) for shp in shapelets]
            row = {f"shapelet_{i+1}_dist": feats[i] for i in range(len(feats))}
            if GROUP_COLS:
                if isinstance(key, tuple):
                    for col, val in zip(GROUP_COLS, key):
                        row[col] = val
                else:
                    row[GROUP_COLS[0]] = key
            feature_rows.append(row)

        shapelet_df = pd.DataFrame(feature_rows)

        fig, ax = plt.subplots(figsize=(10, 5))
        for i, shp in enumerate(shapelets):
            ax.plot(shp, label=f"shapelet_{i+1}")
        ax.set_title("Sample shapelets")
        ax.legend()
        save_fig(fig, "shapelets_sample")

Saved: D:\repositories\personal\xai-spatio-temporal\scripts\figures\shapelets_sample_20260521_102041.png


In [28]:
df_feat = df_deriv.copy()
if shapelet_df is not None:
    if GROUP_COLS:
        df_feat = df_feat.merge(shapelet_df, on=GROUP_COLS, how="left")
    else:
        for col in shapelet_df.columns:
            df_feat[col] = shapelet_df[col].iloc[0]

print("Final shape:", df_feat.shape)
display(df_feat.head())

Final shape: (6000, 136)


,timestamp,date,day_index,time_slot,slot_index,location_id,city,lat,lon,elevation_m,...,heat_index_c__d3,missing_block_flag__d1,missing_block_flag__d2,missing_block_flag__d3,shapelet_1_dist,shapelet_2_dist,shapelet_3_dist,shapelet_4_dist,shapelet_5_dist,shapelet_6_dist
0,2025-09-01 06:00:00+00:00,2025-09-01,0,morning,0,loc_000,Nantes,47.226932,-1.593119,59.391718,...,NaN,NaN,NaN,NaN,3.917797,3.907117,3.919543,3.911124,3.914999,3.91851
1,2025-09-01 12:00:00+00:00,2025-09-01,0,noon,1,loc_000,Nantes,47.226932,-1.593119,59.391718,...,NaN,0.0,NaN,NaN,3.917797,3.907117,3.919543,3.911124,3.914999,3.91851
2,2025-09-01 18:00:00+00:00,2025-09-01,0,evening,2,loc_000,Nantes,47.226932,-1.593119,59.391718,...,NaN,0.0,0.0,NaN,3.917797,3.907117,3.919543,3.911124,3.914999,3.91851
3,2025-09-01 23:00:00+00:00,2025-09-01,0,night,3,loc_000,Nantes,47.226932,-1.593119,59.391718,...,19.97,0.0,0.0,0.0,3.917797,3.907117,3.919543,3.911124,3.914999,3.91851
4,2025-09-02 06:00:00+00:00,2025-09-02,1,morning,0,loc_000,Nantes,47.226932,-1.593119,59.391718,...,-6.24,0.0,0.0,0.0,3.917797,3.907117,3.919543,3.911124,3.914999,3.91851
